# Finite-field MLMD: probe a BaTiO₃ model with a field cycle

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mdi-group/mace-field-tutorial/blob/main/tutorials/macefield/03_finite_field_mlmd_hysteresis.ipynb)


This notebook runs ASE Langevin dynamics using the model trained in notebook 02. It starts from a 135-atom structure in the copied BaTiO₃ reference set and cycles the field parallel to its single labelled `[111]` vector.

The reference data label only one positive field vector. Field reversal, thermal dynamics, and a hysteresis path are outside its training and validation coverage. The saved loop is an exploratory model trajectory, not a validated BaTiO₃ prediction, and its switching field must not be interpreted physically.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

PACKAGE_URL = "https://github.com/mdi-group/mace-field.git"
TUTORIAL_URL = "https://github.com/mdi-group/mace-field-tutorial.git"


def clone_checkout(path, url, branch):
    path = Path(path)
    if not (path / ".git").is_dir():
        if path.exists() and any(path.iterdir()):
            raise RuntimeError(f"Refusing to clone over a non-empty directory: {path}")
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", branch, url, str(path)],
            check=True,
        )


if IN_COLAB:
    REPO = Path("/content/mace-field")
    TUTORIAL_REPO = Path("/content/mace-field-tutorial")
    clone_checkout(REPO, PACKAGE_URL, "clean")
    clone_checkout(TUTORIAL_REPO, TUTORIAL_URL, "main")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--editable", str(REPO)],
        check=True,
    )
else:
    CWD = Path.cwd().resolve()
    candidates = (CWD, *CWD.parents)
    TUTORIAL_REPO = next(
        (path for path in candidates
         if (path / "tutorials/macefield/README.md").is_file()),
        None,
    )
    if TUTORIAL_REPO is None:
        raise RuntimeError(
            "Run from a mace-field-tutorial checkout, or open this notebook in Colab."
        )
    package_candidates = [
        Path(os.environ["MACEFIELD_REPO"]).expanduser().resolve()
        if os.environ.get("MACEFIELD_REPO") else None,
        TUTORIAL_REPO.parent / "mace-field",
        *candidates,
    ]
    REPO = next(
        (path.resolve() for path in package_candidates
         if path is not None and (path / "mace/modules/extensions.py").is_file()),
        None,
    )
    if REPO is None:
        raise RuntimeError(
            "Clone mdi-group/mace-field beside this tutorial checkout, or set MACEFIELD_REPO."
        )
    os.environ["MACEFIELD_REPO"] = str(REPO)

os.environ["MACEFIELD_REPO"] = str(REPO)
sys.path.insert(0, str(REPO))
sys.path.insert(0, str(TUTORIAL_REPO))
print("MACEField source:", REPO)
print("Tutorial assets:", TUTORIAL_REPO)
print("Runtime:", "Google Colab" if IN_COLAB else "local Jupyter")


**Colab checkpoint note:** each notebook can start in a fresh runtime. Run notebook 02 and upload its generated `.model` file when prompted here, or set `MACEFIELD_MODEL_PATH` to a checkpoint already available in this runtime.


## What the trajectory means

A field-driven trajectory depends on its starting structure, temperature, sweep rate, and model. This short run shows how to update the MACEField calculator during ASE dynamics, save the trajectory and a CSV record, and plot polarization against the applied field.

The trained model has only one field vector in its reference data. Even a symmetric cycle with the same maximum field magnitude probes zero and negative fields that were not labelled. The trajectory is useful for workflow exploration; it cannot establish a physical hysteresis loop or coercive field.

In [ ]:
import csv
import json
import os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from ase import units
from ase.md.langevin import Langevin
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution, Stationary
from ase.io import read, write
from mace.calculators import MACECalculator
import torch

TUTORIAL_DIR = TUTORIAL_REPO / "tutorials/macefield"
DATA = TUTORIAL_DIR / "data"
TRAINED_MODEL = TUTORIAL_DIR / "runs/bto_macefield_training/MACEField-BaTiO3-reference_run-23.model"
MODEL = Path(os.environ.get("MACEFIELD_MODEL_PATH", str(TRAINED_MODEL))).expanduser()
if not MODEL.is_absolute():
    MODEL = TUTORIAL_REPO / MODEL
if not MODEL.is_file() and IN_COLAB:
    from google.colab import files
    print("Upload the .model checkpoint produced by notebook 02, or set MACEFIELD_MODEL_PATH first.")
    uploaded = files.upload()
    model_uploads = [(name, content) for name, content in uploaded.items() if name.endswith(".model")]
    if model_uploads:
        upload_name, upload_bytes = model_uploads[0]
        MODEL = Path("/content") / Path(upload_name).name
        MODEL.write_bytes(upload_bytes)

if not MODEL.is_file():
    raise FileNotFoundError(
        f"Model not found: {MODEL}. Run notebook 02, upload its .model checkpoint in Colab, "
        "or set MACEFIELD_MODEL_PATH."
    )
DEVICE = os.environ.get("MACEFIELD_DEVICE",
                        "cuda" if torch.cuda.is_available() else "cpu")
train = read(DATA / "macefield_batio3_reference_train.extxyz", index=":")
print("model:", MODEL, "| device:", DEVICE, "| training frames:", len(train))


## Choose a structure and field direction from the reference set

The field direction below comes directly from the ExtXYZ label. The initial structure is the training frame with the smallest polarization projected along that direction. All 135 atoms evolve under the ASE thermostat.

In [ ]:
training_fields = np.asarray([a.info["REF_electric_field"] for a in train], dtype=float)
reference_field = np.mean(training_fields, axis=0)
field_direction = reference_field / np.linalg.norm(reference_field)
training_p_parallel = np.asarray([
    np.dot(a.info["REF_polarization"], field_direction) for a in train
])
start_index = int(np.argmin(training_p_parallel))
atoms = train[start_index].copy()
print("starting frame:", start_index)
print("formula:", atoms.get_chemical_formula(), "| atoms:", len(atoms))
print("cell volume (Å³):", atoms.get_volume())
print("reference field (V/Å):", reference_field)
print("starting P parallel (e/Å²):", float(training_p_parallel[start_index]))


## Check the field coverage

All training frames carry the same positive vector. The diagnostic below is intentionally visible before the trajectory cell: it shows the supported field sample and states that the reversal branch is extrapolative. Keep the default cycle amplitude at or below the labelled vector magnitude.

In [ ]:
unique_training_fields = np.unique(np.round(training_fields, decimals=10), axis=0)
print("unique labelled field vectors (V/Å):\n", unique_training_fields)
print("labelled field-vector norm (V/Å):", float(np.linalg.norm(reference_field)))
print("WARNING: the negative and zero portions of the cycle are not covered by the labels.")
print("Do not read this trajectory as a validated hysteresis or coercive-field prediction.")


## Set the field cycle and MD protocol

The field is updated between constant-field MD segments. This example sweeps the labelled vector from negative to positive and back, with no field magnitude larger than the reference vector. The negative and zero fields remain out of the training domain. Increase the number of points or steps only for workflow exploration, not to claim convergence of a physical loop.

In [ ]:
FIELD_SCALE_MAX = float(os.environ.get("MACEFIELD_FIELD_SCALE", "1.0"))
N_FIELD_POINTS = 9
STEPS_PER_FIELD = 5
DT_FS = 0.5
TEMPERATURE_K = 25.0
FRICTION_PER_FS = 0.02
EQUILIBRATION_STEPS = 20
SEED = 23

up_scale = np.linspace(-FIELD_SCALE_MAX, FIELD_SCALE_MAX, N_FIELD_POINTS)
down_scale = up_scale[-2::-1]
field_scale_cycle = np.concatenate([up_scale, down_scale])
field_cycle = field_scale_cycle[:, None] * reference_field[None, :]
print("field points:", len(field_cycle), "| total cycle time (fs):",
      len(field_cycle) * STEPS_PER_FIELD * DT_FS)
print("field scale range:", float(field_scale_cycle.min()), "to",
      float(field_scale_cycle.max()))


In [ ]:
calc = MACECalculator(
    model_paths=str(MODEL), model_type="MACEField", device=DEVICE,
    default_dtype="float64", electric_field=field_cycle[0],
    compute_polarization=True, compute_becs=False,
    compute_polarizability=False,
)
atoms.calc = calc
rng = np.random.default_rng(SEED)
MaxwellBoltzmannDistribution(atoms, temperature_K=TEMPERATURE_K, rng=rng)
Stationary(atoms)
dyn = Langevin(
    atoms, timestep=DT_FS * units.fs,
    temperature_K=TEMPERATURE_K,
    friction=FRICTION_PER_FS / units.fs,
    rng=rng,
)
dyn.run(EQUILIBRATION_STEPS)
print("equilibrated structure | potential energy (eV):", atoms.get_potential_energy())


## Run the cycle and save predictions

The ExtXYZ and CSV contain model predictions for this exact short protocol. The data source and model path are recorded next to the outputs by the notebook's printed settings; the files do not contain reference DFT labels for these new MD frames.

In [ ]:
OUTPUT_DIR = Path(os.environ.get(
    "MACEFIELD_MD_OUTPUT_DIR",
    str(TUTORIAL_DIR / "runs/mlmd_reference_data_field_cycle"),
)).expanduser()
if not OUTPUT_DIR.is_absolute():
    OUTPUT_DIR = REPO / OUTPUT_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
trajectory_path = OUTPUT_DIR / "bto_reference_data_field_cycle.extxyz"
csv_path = OUTPUT_DIR / "bto_reference_data_field_cycle.csv"
for path in (trajectory_path, csv_path):
    if path.exists():
        path.unlink()

records = []
for istep, (scale, field) in enumerate(zip(field_scale_cycle, field_cycle)):
    calc.electric_field = field.tolist()
    calc.reset()
    dyn.run(STEPS_PER_FIELD)
    energy = atoms.get_potential_energy()
    polarization = np.asarray(calc.results["polarization"], dtype=float)
    row = {
        "field_index": istep,
        "direction": "up" if istep < len(up_scale) else "down",
        "time_fs": EQUILIBRATION_STEPS * DT_FS +
                   (istep + 1) * STEPS_PER_FIELD * DT_FS,
        "field_scale": float(scale),
        "Ex_V_per_A": float(field[0]),
        "Ey_V_per_A": float(field[1]),
        "Ez_V_per_A": float(field[2]),
        "E_parallel_V_per_A": float(np.dot(field, field_direction)),
        "P_parallel_e_per_A2": float(np.dot(polarization, field_direction)),
        "energy_eV": float(energy),
    }
    records.append(row)
    frame = atoms.copy()
    frame.calc = None
    for key in ("REF_energy", "REF_stress", "REF_polarization",
                "REF_polarizability", "REF_electric_field"):
        frame.info.pop(key, None)
    for key in ("REF_forces", "REF_becs"):
        frame.arrays.pop(key, None)
    frame.info["electric_field"] = field.tolist()
    frame.info["MACEField_P_e_per_A2"] = polarization.tolist()
    frame.info["field_cycle_direction"] = row["direction"]
    write(trajectory_path, frame, format="extxyz", append=(istep > 0))

with csv_path.open("w", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=list(records[0]))
    writer.writeheader()
    writer.writerows(records)
print("trajectory:", trajectory_path)
print("CSV:", csv_path)
print("frames:", len(records))


## Plot polarization against field

The conversion is (1\,e/\text{Å}^2 = 1602.176634\,\mu\text{C}/\text{cm}^2). The plotted values are projections along the labelled `[111]` field direction. The signed response and the sweep direction are retained.

In [ ]:
loop = {key: np.asarray([row[key] for row in records]) for key in records[0]}
P_UC_CM2 = 1602.176634
is_up = np.array([row["direction"] == "up" for row in records])
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(loop["E_parallel_V_per_A"][is_up],
             loop["P_parallel_e_per_A2"][is_up] * P_UC_CM2,
             "o-", ms=4, label="increasing field")
axes[0].plot(loop["E_parallel_V_per_A"][~is_up],
             loop["P_parallel_e_per_A2"][~is_up] * P_UC_CM2,
             "o-", ms=4, label="decreasing field")
axes[0].set(xlabel="field projection (V/Å)", ylabel="polarization projection (µC/cm²)",
            title="Exploratory finite-rate model response")
axes[0].legend()
axes[1].plot(loop["time_fs"], loop["P_parallel_e_per_A2"] * P_UC_CM2,
             color="#7b4f9d")
axes[1].set(xlabel="time (fs)", ylabel="polarization projection (µC/cm²)",
            title="Model polarization along the trajectory")
for ax in axes:
    ax.grid(alpha=0.22)
plt.tight_layout()
plt.show()
loop_area = float(np.sum(
    0.5 * (loop["P_parallel_e_per_A2"][1:] + loop["P_parallel_e_per_A2"][:-1])
    * np.diff(loop["E_parallel_V_per_A"])
))
print(f"signed trajectory integral ∮ P·dE: {loop_area:.6g} (e/Å²)(V/Å)")


## Interpretation and next data needed

This finite-rate curve records the model's response to the chosen thermostat, timestep, starting configuration, and field schedule. Since the source set has one positive field vector, it does not validate the reverse branch or switching. Do not report its area or apparent switching field as a BaTiO₃ property.

A predictive hysteresis study needs reference calculations at both field polarities and multiple magnitudes, structures along the switching path, thermal configurations, an independent test split, and checks of energy–force and energy–polarization derivative agreement over the visited domain. It also needs slower sweeps and larger supercells for convergence.